In [4]:
import math
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.optimize as optimize
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.io import imread, imsave

np.random.seed(52)

# исходные параметры
alpha_level = 0.05
parts = 10
n_obs = 100
x_marks = np.arange(parts)
obs = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
base = np.ones(parts)

# аппроксимация функции Колмогорова

def kolmogorov_tail(z):
    s = 1.0
    for t in range(1, 1001):
        s += 2 * (-1) ** t * np.exp(-2 * (t ** 2) * (z ** 2))
    return s

# целевая функция для подбора параметров нормального закона

def loss_normal(arg):
    mu, sigma = arg
    cdf_vals = stats.norm.cdf(x_marks[1:], loc=mu, scale=sigma)

    center_block = 1
    for pos in range(8):
        center_block *= (cdf_vals[pos + 1] - cdf_vals[pos]) ** obs[pos + 1]

    left_block = cdf_vals[0] ** obs[0]
    right_block = (1 - cdf_vals[-1]) ** obs[-1]
    return -(left_block * center_block * right_block)

# блок для равномерного распределения
exp_uniform = 10 * base
chi2_uniform = float(np.sum((obs - exp_uniform) ** 2 / exp_uniform))
p_uniform_chi2 = float(1 - stats.chi2(parts - 1).cdf(chi2_uniform))

print('Равномерная модель R(0,10), критерий хи-квадрат')
print(f'chi^2 = {chi2_uniform:.6f}; p-value = {p_uniform_chi2:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_uniform_chi2 > alpha_level else 'отвергается')

emp_cdf = np.array([np.sum(obs[:i]) for i in range(len(obs) + 1)]) / n_obs
theory_uniform_cdf = np.array([np.sum(base[:i]) for i in range(len(base))]) / 10

ks_uniform = float(
    np.sqrt(n_obs) * np.max([
        max(
            np.abs(emp_cdf[i] - theory_uniform_cdf[i]),
            np.abs(emp_cdf[i + 1] - theory_uniform_cdf[i])
        )
        for i in range(parts)
    ])
)
p_uniform_ks = 1 - kolmogorov_tail(ks_uniform)

print()
print('Равномерная модель R(0,10), критерий Колмогорова')
print(f'Δ = {ks_uniform:.2f}; p-value = {p_uniform_ks:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_uniform_ks > alpha_level else 'отвергается')

# блок для нормального распределения
best = optimize.differential_evolution(
    func=loss_normal,
    bounds=[(0, 10), (0, 10)],
    maxiter=10000,
)

mu_hat = float(best.x[0])
sigma_hat = float(best.x[1])
cdf_vals = stats.norm.cdf(x_marks[1:], loc=mu_hat, scale=sigma_hat)

exp_normal = [n_obs * cdf_vals[0]]
for pos in range(8):
    exp_normal.append(n_obs * (cdf_vals[pos + 1] - cdf_vals[pos]))
exp_normal.append(n_obs * (1 - cdf_vals[-1]))
exp_normal = np.array(exp_normal)

chi2_normal = float(np.sum((obs - exp_normal) ** 2 / exp_normal))
p_normal_chi2 = float(1 - stats.chi2(parts - 2 - 1).cdf(chi2_normal))

print()
print('Нормальная модель N(theta1, theta2^2), критерий хи-квадрат')
print(f'theta1 = {mu_hat:.6f}; theta2 = {sigma_hat:.6f}')
print(f'chi^2 = {chi2_normal:.6f}; p-value = {p_normal_chi2:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_normal_chi2 > alpha_level else 'отвергается')

BOOTSTRAP_SIZE = 50000
sample_rebuilt = []
for pos in range(parts):
    sample_rebuilt += [pos] * obs[pos]
sample_rebuilt = np.array(sample_rebuilt)

mean_hat = np.mean(sample_rebuilt)
std_hat = np.std(sample_rebuilt, ddof=1)

# функция распределения для нормального закона через erf

def normal_cdf(x, m, s):
    return 0.5 * (1 + math.erf((x - m) / (np.sqrt(2) * s)))

ks_normal = np.max([
    np.sqrt(n_obs) * max(
        np.abs(normal_cdf(x_marks[i], mean_hat, std_hat) - emp_cdf[i]),
        np.abs(normal_cdf(x_marks[i], mean_hat, std_hat) - emp_cdf[i + 1])
    )
    for i in range(10)
])


def bootstrap(size):
    out = []
    law = stats.norm(loc=mean_hat, scale=std_hat)

    for _ in range(size):
        draw = np.sort(law.rvs(size=n_obs))
        draw_emp_cdf = [i / n_obs for i in range(n_obs + 1)]
        draw_mean = np.mean(draw)
        draw_std = np.std(draw, ddof=1)

        out.append(np.max([
            np.sqrt(n_obs) * max(
                np.abs(normal_cdf(draw[j], draw_mean, draw_std) - draw_emp_cdf[j]),
                np.abs(normal_cdf(draw[j], draw_mean, draw_std) - draw_emp_cdf[j + 1])
            )
            for j in range(len(draw))
        ]))
    return out

print()
print('Нормальная модель N(theta1, theta2^2), критерий Колмогорова')
print(f'Δ = {ks_normal:.6f}')

boot_stats = bootstrap(BOOTSTRAP_SIZE)
p_normal_ks = np.mean(np.array(boot_stats) >= ks_normal)

print(f'p-value: {p_normal_ks:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_normal_ks > alpha_level else 'отвергается')



Равномерная модель R(0,10), критерий хи-квадрат
chi^2 = 16.400000; p-value = 0.0590
Решение по H0: нет оснований для отклонения гипотезы

Равномерная модель R(0,10), критерий Колмогорова
Δ = 1.40; p-value = 0.0397
Решение по H0: отвергается

Нормальная модель N(theta1, theta2^2), критерий хи-квадрат
theta1 = 5.292873; theta2 = 2.675816
chi^2 = 9.799736; p-value = 0.2002
Решение по H0: нет оснований для отклонения гипотезы

Нормальная модель N(theta1, theta2^2), критерий Колмогорова
Δ = 1.002094
p-value: 0.0153
Решение по H0: отвергается
